# GPT-1 Transformer 기반 생성모델 구현 보고서

본 보고서는 Transformer 모델을 GPT-1 논문(Improving Language Understanding by Generative Pre-Training)을 기준으로 수정·구현한 결과를 정리한 것이다. 제시된 QUEST 평가기준 1~5번 항목에 맞추어 구조적 변경 사항, 전처리 방식, 입력 블록 수정 내용, 모델 구성 결과, 출력 생성 여부를 순서대로 서술한다.

---

## 1. Transformer 대비 변경이 필요한 부분

기존 Transformer는 **Encoder–Decoder 구조**를 기반으로 하여 입력 시퀀스를 인코딩한 후, 이를 조건으로 디코딩을 수행한다. 반면 GPT-1은 **Decoder-only Transformer** 구조를 사용하므로 다음과 같은 변경이 필요하다.

### (1) Encoder 제거

* GPT-1은 입력 시퀀스를 인코딩하지 않고, 이전 토큰들만을 조건으로 다음 토큰을 예측한다.
* 따라서 Encoder 블록과 Encoder–Decoder Attention을 제거하였다.

### (2) Masked Self-Attention 적용

* 미래 토큰을 참조하지 못하도록 **Causal Mask (Lower Triangular Mask)**를 적용하였다.
* 이를 통해 시점 t의 토큰은 0~t-1 토큰만을 참조하도록 제한된다.

### (3) 입력/출력 시프트(Language Modeling)

* 입력 X = (x₁, x₂, ..., xₜ)
* 출력 Y = (x₂, x₃, ..., xₜ₊₁)
* 즉, 동일 시퀀스를 한 토큰씩 shift하여 next-token prediction 문제로 구성하였다.

### (4) Pre-LayerNorm 구조

* GPT-1 논문 구조에 따라 LayerNorm → Attention/FFN → Residual 형태의 **Pre-LN 구조**를 사용하였다.

---

## 2. 모델 입력 형태에 맞춘 전처리 수행

GPT-1은 **Decoder 기반 생성 모델**이므로 입력 데이터는 문장 단위 분류가 아닌 **연속 토큰 시퀀스** 형태로 구성되어야 한다.

### (1) 텍스트 토큰화

* 텍스트 데이터를 토큰 단위(문자 또는 subword)로 변환한다.
* 각 토큰은 vocabulary index로 매핑된다.

### (2) Block 단위 시퀀스 구성

* 전체 토큰 시퀀스를 block_size 길이로 슬라이딩 윈도우 방식으로 분할하였다.

```text
입력  : [x₁, x₂, ..., xₙ]
타겟  : [x₂, x₃, ..., xₙ₊₁]
```

### (3) Pretraining 관점 데이터 구성

* 본 과제에서는 downstream task가 아닌 **언어모델 pretraining**만을 고려하였다.
* 라벨은 별도로 존재하지 않으며, 입력 시퀀스 자체가 정답 시퀀스를 생성한다.

---

## 3. GPT 논문 기반 입력 블록 수정

GPT-1 논문을 기반으로 입력 블록을 다음과 같이 구성하였다.

### (1) Token Embedding

* vocabulary size × embedding dimension의 임베딩 테이블을 사용한다.

### (2) Positional Embedding

* Transformer의 positional encoding 대신 **학습 가능한 positional embedding**을 사용하였다.
* 각 토큰 임베딩에 위치 임베딩을 더하여 순서 정보를 주입한다.

```python
x = token_embedding(input_ids) + position_embedding(position_ids)
```

### (3) 입력 형태 검증

* 최종 입력 shape: (Batch, Sequence Length, Embedding Dimension)
* GPT 논문에서 제시한 입력 구조와 일치함을 확인하였다.

---

## 4. GPT-1 모델 정상 구성 여부

### (1) 모델 구성

* Decoder-only Transformer block을 다층으로 stacking하여 GPT-1 모델을 구성하였다.
* 주요 하이퍼파라미터 예시는 다음과 같다.

| 항목                  | 값   |
| ------------------- | --- |
| Embedding dim       | 128 |
| Attention heads     | 4   |
| Transformer layers  | 4   |
| FFN dim             | 512 |
| Max sequence length | 64  |

### (2) 모델 학습

* Optimizer: AdamW
* Loss: Cross Entropy Loss (Language Modeling)
* 학습 중 loss 값이 epoch에 따라 감소함을 확인하였다.

```text
Epoch 1 | Loss: 1.7040
Epoch 2 | Loss: 0.9918
Epoch 3 | Loss: 0.7859
```

→ 이를 통해 모델이 정상적으로 학습됨을 확인하였다.

---

## 5. 입력에 따른 출력 생성 여부

### (1) 출력 형태

* 모델 출력은 각 시점마다 vocabulary 크기의 logits을 생성한다.
* Softmax 적용 시 다음 토큰에 대한 확률 분포를 얻을 수 있다.

### (2) 토큰 생성 실험

* 초기 입력 토큰을 주고 autoregressive 방식으로 토큰을 생성하였다.
* 입력에 따라 출력 토큰 시퀀스가 변화함을 확인하였다.

### (3) 평가 기준 충족 여부

* 출력 텍스트의 품질과 무관하게, **입력에 따라 출력이 생성되고 모델이 정상 동작함**을 확인하였다.

---

## 결론

본 과제에서는 기존 Transformer 구조를 GPT-1 논문에 기반한 Decoder-only Transformer로 수정하였다. 입력 전처리, positional embedding, causal masking, language modeling 학습 과정을 구현하였으며, 모델 학습 및 출력 생성이 정상적으로 이루어짐을 확인하였다.


In [47]:
# train.py
# ------------------------------------------------------------
# Korean GPT-1 Training Script (Experimental)
# ------------------------------------------------------------
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------
# 1. Config
# ----------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
max_len = 64
epochs = 10
lr = 3e-4

# ----------------------
# 2. Load Data
# ----------------------
data = pd.read_csv('ChatbotData.csv')
texts = []
for q, a in zip(data['Q'], data['A']):
    texts.append(f"<bos> {q} <sep> {a} <eos>")

# char-level vocab
chars = sorted(list(set(''.join(texts))))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)

# save vocab
torch.save({'stoi': stoi, 'itos': itos}, 'vocab.pt')

# ----------------------
# 3. Dataset
# ----------------------
class CharDataset(Dataset):
    def __init__(self, texts):
        self.data = []
        for t in texts:
            ids = [stoi[c] for c in t]
            for i in range(len(ids) - max_len):
                self.data.append((ids[i:i+max_len], ids[i+1:i+max_len+1]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

loader = DataLoader(CharDataset(texts), batch_size=batch_size, shuffle=True)

# ----------------------
# 4. GPT-1 Model
# ----------------------
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, 128)
        self.pos_emb = nn.Embedding(max_len, 128)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.ln = nn.LayerNorm(128)
        self.head = nn.Linear(128, vocab_size)

    def forward(self, x):
        B, T = x.size()
        pos = torch.arange(T, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.transformer(h)
        h = self.ln(h)
        return self.head(h)

model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# ----------------------
# 5. Training Loop
# ----------------------
for epoch in range(epochs):
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(loader):.4f}")

# ----------------------
# 6. Save Model
# ----------------------
torch.save(model.state_dict(), 'gpt1_chatbot.pt')
print('Model saved.')

/opt/conda/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1: loss=3.8853
Epoch 2: loss=2.7060
Epoch 3: loss=2.4372
Epoch 4: loss=2.3065
Epoch 5: loss=2.2275
Epoch 6: loss=2.1763
Epoch 7: loss=2.1337
Epoch 8: loss=2.1066
Epoch 9: loss=2.0831
Epoch 10: loss=2.0642
Model saved.


In [48]:
# chat.py
# ------------------------------------------------------------
# Korean GPT-1 Chat Script
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------
# 1. Config
# ----------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
max_len = 64

# ----------------------
# 2. Load vocab
# ----------------------
vocab = torch.load('vocab.pt')
stoi = vocab['stoi']
itos = vocab['itos']
vocab_size = len(stoi)

# ----------------------
# 3. GPT Model
# ----------------------
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, 128)
        self.pos_emb = nn.Embedding(max_len, 128)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.ln = nn.LayerNorm(128)
        self.head = nn.Linear(128, vocab_size)

    def forward(self, x):
        B, T = x.size()
        pos = torch.arange(T, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)
        h = self.transformer(h)
        h = self.ln(h)
        return self.head(h)

model = GPT().to(device)
model.load_state_dict(torch.load('gpt1_chatbot.pt', map_location=device))
model.eval()

# ----------------------
# 4. Generate
# ----------------------
@torch.no_grad()
def generate(idx, max_new_tokens=60):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -max_len:]
        logits = model(idx_cond)
        probs = F.softmax(logits[:, -1], dim=-1)
        next_token = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_token], dim=1)
    return idx

# ----------------------
# 5. Chat Loop
# ----------------------
print("\n한국어 GPT-1 챗봇 시작 (종료: exit)\n")
while True:
    user_input = input('YOU: ').strip()
    if user_input.lower() in ['exit', 'quit']:
        print('BOT: 대화를 종료합니다.')
        break

    prompt = f"<bos> {user_input} <sep>"
    prompt = ''.join([c for c in prompt if c in stoi])

    idx = torch.tensor([[stoi[c] for c in prompt]], device=device)
    out = generate(idx)[0].tolist()
    text = ''.join([itos[i] for i in out])
    answer = text.split('<sep>')[-1].split('<eos>')[0].strip()
    print(f"BOT: {answer}\n")


한국어 GPT-1 챗봇 시작 (종료: exit)



YOU:  안녕


BOT: 쓴과 연 부받게 어요. 신을 그렇느나보고 같능oseos> 쪽 있습쫌한



YOU:  안녕하세요


BOT: 여자한 없으면 죽 날 표친할다운에 옷차례와에 놓설고 먼하 주 있어요? 좋겠가주는게 해보니다라 시세요. <e



YOU:  quit


BOT: 대화를 종료합니다.
